# OpenStreetMap

In [5]:
import geopandas as gpd

## Carga de datos

In [ ]:
path_pbf = 'osm/andalucia-260305.osm.pbf'

# 1. Extraer el polígono de la Provincia de Cádiz (Admin Level 6)
provincia_cadiz = gpd.read_file(
    path_pbf,
    engine='pyogrio',
    layer='multipolygons',
    where="admin_level = '6' AND name = 'Cádiz'"
)

# 2. Extraer todos los municipios (Admin Level 8) de Andalucía
municipios_and = gpd.read_file(
    path_pbf,
    engine='pyogrio',
    layer='multipolygons',
    where="admin_level = '8'"
)

In [7]:
provincia_cadiz

,osm_id,osm_way_id,name,type,aeroway,amenity,admin_level,barrier,boundary,building,...,man_made,military,natural,office,place,shop,sport,tourism,other_tags,geometry
0,349017,None,Cádiz,boundary,None,None,6,None,administrative,None,...,None,None,None,None,province,None,None,None,"""ISO3166-2""=>""ES-CA"",""ine:provincia""=>""11"",""na...","MULTIPOLYGON (((-5.27354 36.292, -5.2736 36.29..."


In [8]:
# Es vital que ambos tengan el mismo Sistema de Referencia (CRS)
# Generalmente OSM viene en EPSG:4326
municipios_and = municipios_and.to_crs(provincia_cadiz.crs)

# Realizamos la unión espacial
# 'inner' para quedarnos solo con los que coinciden
# 'within' asegura que el municipio esté dentro de la provincia
municipios_cadiz = gpd.sjoin(municipios_and, provincia_cadiz, predicate='within', how='inner')

# Limpiar columnas duplicadas tras el join (opcional)
municipios_cadiz = municipios_cadiz[['name_left', 'geometry', 'admin_level_left']]
municipios_cadiz.columns = ['name', 'geometry', 'admin_level']

In [9]:
lista_limites = []

for idx, row in municipios_cadiz.iterrows():
    # .bounds devuelve (minx, miny, maxx, maxy)
    minx, miny, maxx, maxy = row['geometry'].bounds
    
    lista_limites.append({
        'municipio': row['name'],
        'bbox': [minx, miny, maxx, maxy]
    })

# Ejemplo de visualización rápida
print(f"Total municipios encontrados en Cádiz: {len(municipios_cadiz)}")
print(f"Límites de {lista_limites[0]['municipio']}: {lista_limites[0]['bbox']}")

Total municipios encontrados en Cádiz: 45
Límites de Paterna de Rivera: [-5.8954597, 36.4971526, -5.8285846, 36.5382019]


In [10]:
lista_limites

[{'municipio': 'Paterna de Rivera',
  'bbox': [-5.8954597, 36.4971526, -5.8285846, 36.5382019]},
 {'municipio': 'Olvera',
  'bbox': [-5.417881, 36.8701671, -5.0956924, 37.0373629]},
 {'municipio': 'El Puerto de Santa María',
  'bbox': [-6.323047, 36.5274471, -6.1396142, 36.7189243]},
 {'municipio': 'Prado del Rey',
  'bbox': [-5.599987, 36.7330958, -5.5001346, 36.8359545]},
 {'municipio': 'Jimena de la Frontera',
  'bbox': [-5.5880574, 36.3437703, -5.3441085, 36.5377821]},
 {'municipio': 'Jerez de la Frontera',
  'bbox': [-6.3061225, 36.5162096, -5.4557904, 36.8637895]},
 {'municipio': 'Medina Sidonia',
  'bbox': [-6.0725169, 36.2150926, -5.616249, 36.5354686]},
 {'municipio': 'La Línea de la Concepción',
  'bbox': [-5.3708792, 36.1528423, -5.3105786, 36.2465604]},
 {'municipio': 'Puerto Serrano',
  'bbox': [-5.5506207, 36.8934473, -5.4200851, 37.0524453]},
 {'municipio': 'Puerto Real',
  'bbox': [-6.2591065, 36.4557889, -5.9801308, 36.5993865]},
 {'municipio': 'Algar',
  'bbox': [-5.6

Ya tenemos una lista con los municipios de Cádiz y su bounding box correspondiente

# Copernicus